In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

In [0]:
%sql
DROP TABLE IF EXISTS default.courses;
DROP TABLE IF EXISTS default.courses_autoloader_sink;


In [0]:
import os

test_file_to_delete = "/Volumes/workspace/default/volume/courses_part2.json"
autoloader_checkpoint = "/Volumes/workspace/default/volume/_chk/autoloader_courses"

# --- 1. Tabellen löschen (Unity Catalog Pfad nutzen) ---
spark.sql("DROP TABLE IF EXISTS workspace.default.courses_autoloader_sink")
print("✅ Sink-Tabelle 'workspace.default.courses_autoloader_sink' gelöscht.")

# --- 2. Test-Datei im Volume löschen (Saubere Prüfung) ---
if os.path.exists(test_file_to_delete):
    dbutils.fs.rm(test_file_to_delete)
    print(f"✅ Test-Datei '{test_file_to_delete}' wurde entfernt.")
else:
    print(f"ℹ️ Test-Datei '{test_file_to_delete}' war nicht vorhanden.")

# --- 3. Alle Checkpoints löschen (Wichtig für Auto Loader Reset) ---
if os.path.exists(autoloader_checkpoint):
    dbutils.fs.rm(autoloader_checkpoint, recurse=True)
    print(f"✅ Auto Loader Checkpoint unter '{autoloader_checkpoint}' gelöscht.")
else:
    print("ℹ️ Kein Checkpoint gefunden, sauberer Start möglich.")

print("\n🚀 System ist vollständig zurückgesetzt. Auto Loader startet nun bei 'Stunde Null'.")

In [0]:
dbutils.fs.rm("/Volumes/workspace/default/volume/_chk/autoloader_courses", recurse=True)

# Vorbereitung: Datei einlesen und als Tabelle abspeichern

In [0]:
%python
dataset_school = "/Volumes/workspace/default/volume"

all_files = dbutils.fs.ls(dataset_school)
json_files = [f for f in all_files if f.name.endswith(".json")]

display(json_files)

# Auto Loader

In [0]:
# Pfad zum Ordner, den der Auto Loader überwachen soll
source_path = "/Volumes/workspace/default/volume/"

# Pfad für den Checkpoint, damit sich der Auto Loader merkt, was er gelesen hat
checkpoint_path = "/Volumes/workspace/default/volume/_chk/autoloader_courses"

# Ziel-Tabelle
target_table = "courses_autoloader_sink"

In [0]:
# Auto Loader Stream definieren und starten
autoloader_stream = (
  spark.readStream
    .format("cloudFiles")  # Auto Loader aktivieren
    .option("cloudFiles.format", "json")
    # Schema-Inferenz: Der Auto Loader speichert hier das erkannte Tabellen-Schema
    .option("cloudFiles.schemaLocation", f"{checkpoint_path}/_schema") 
    .option("pathGlobFilter", "courses*.json") # Nur Dateien verarbeiten, die mit "courses" beginnen
    .load(source_path)
  .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true") # Erlaubt das Hinzufügen neuer Spalten
    .trigger(availableNow=True) # Notwendig für die Free Edition
    .toTable(target_table)
)

autoloader_stream.awaitTermination() # blockiert die Ausführung der Notebook-Zelle und wartet, bis der Streaming-Job beendet ist.

print(f"✅ Auto Loader hat die Daten erfolgreich in die Tabelle '{target_table}' geladen.")

In [0]:
%sql
SELECT * FROM workspace.default.courses_autoloader_sink;

In [0]:
# Inhalt für unsere neue Datei
json_data = """
[
  {"course_id": 301, "title": "Advanced SQL", "instructor": "Dr. Codd", "category": "Technology", "price": 300},
  {"course_id": 302, "title": "Data Warehousing", "instructor": "Dr. Codd", "category": "Technology", "price": 400},
  {"course_id": 104, "title": "Applied Ethics", "instructor": "Dr. Kim", "category": "Philosophy", "price": 250}
]
"""

# Pfad und Dateiname
file_path = "/Volumes/workspace/default/volume/courses_part2.json"

# Die neue Datei erstellen
dbutils.fs.put(file_path, json_data, overwrite=True)

print(f"✅ Neue Datei '{file_path}' wurde erfolgreich erstellt.")

In [0]:
%sql
-- 1. Führe die Auto-Loader-Zelle (Code-Block 9) erneut aus.
--    Der Stream wird nur die neue Datei "courses_part2.json" verarbeiten.

-- 2. Überprüfe das Endergebnis.
--    Die Tabelle enthält jetzt die Daten aus beiden Dateien.
SELECT * FROM workspace.default.courses_autoloader_sink ORDER BY course_id;

In [0]:
%sql
DESCRIBE HISTORY courses_autoloader_sink

 #Aufräumen

In [0]:
%sql
DROP TABLE IF EXISTS default.courses_autoloader_sink;